# Notebook 15: Dimensionality Reduction – PCA, t-SNE, UMAP
**Part 15/30 – ML Mastery Series for Python Experts**

## Why Reduce Dimensions? – Curse, Noise, Speed & Interpretability

You already know high dimensions are trouble — now let's learn how to tame them, visualize the invisible, and sometimes even make models faster and better.

- **Curse of Dimensionality**: As dimensions grow, data becomes sparse; distance metrics lose discriminative power, and models struggle to generalize
- **Multicollinearity**: High-dimensional data often contains redundant features; dimensionality reduction decorrelates and removes redundancy
- **Visualization**: Humans can only see 2D/3D; reduction projects high-D data into interpretable visual spaces
- **Faster Training & Inference**: Fewer features → smaller matrices → faster computation and lower memory footprint
- **Noise Removal**: Low-variance components often capture noise; discarding them can improve signal-to-noise ratio
- **Better Generalization**: Sometimes the "manifold" structure is lower-dimensional; finding it helps models focus on true patterns
- **Compression**: Store and transmit data efficiently by keeping only the most informative dimensions
- **Information Loss Risk**: Aggressive reduction can discard predictive signal — always validate downstream impact

## Learning Objectives

By the end of this notebook, you will:

- Understand PCA's variance maximization objective and how principal components capture orthogonal directions of maximum variance
- Interpret PCA loadings to see which original features contribute to each component
- Use PCA for preprocessing, compression, and reconstruction with controlled information loss
- Visualize high-dimensional data using t-SNE while understanding its local structure preservation
- Apply UMAP for faster, more scalable embeddings that balance local and global structure
- Compare local vs global structure preservation across all three methods
- Tune critical hyperparameters: perplexity (t-SNE), n_neighbors & min_dist (UMAP)
- Evaluate reduction quality using reconstruction error (PCA) or trustworthiness/continuity metrics
- Know when to use each method: PCA for linear preprocessing, t-SNE for visualization, UMAP for scalable exploration

## 📉 1. PCA – Linear Variance Maximization

PCA finds orthogonal axes (principal components) that maximize variance. The first PC captures the direction of maximum variance, the second captures the next orthogonal direction, and so on.

**Mathematical intuition**: Maximize $w^T \Sigma w$ subject to $w^T w = 1$, where $\Sigma$ is the covariance matrix. This leads to eigenvectors of $\Sigma$ as principal components.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configure plotting
%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load high-dimensional digits dataset (8x8 images = 64 features)
digits = load_digits()
X, y = digits.data, digits.target
print(f"Original data shape: {X.shape} — {X.shape[1]} features, {len(np.unique(y))} classes")

# Always scale before PCA (variance-based method)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA preserving 95% of variance (let sklearn pick components)
pca_full = PCA(n_components=0.95, random_state=42)
X_pca_full = pca_full.fit_transform(X_scaled)
print(f"Components for 95% variance: {pca_full.n_components_}")

# Also fit PCA with all components to see full variance profile
pca_all = PCA()
pca_all.fit(X_scaled)
print(f"Total possible components: {len(pca_all.explained_variance_ratio_)}")

In [ ]:
# Plot cumulative explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of individual variance ratios
ax1.bar(range(1, 21), pca_all.explained_variance_ratio_[:20], alpha=0.7, color='steelblue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Variance Explained by First 20 Components')
ax1.set_xticks(range(1, 21, 2))

# Cumulative variance line
cumulative = np.cumsum(pca_all.explained_variance_ratio_)
ax2.plot(range(1, len(cumulative)+1), cumulative, 'o-', markersize=3, color='darkred')
ax2.axhline(y=0.95, color='r', linestyle='--', alpha=0.7, label='95% threshold')
ax2.axvline(x=pca_full.n_components_, color='g', linestyle='--', alpha=0.7, label=f'{pca_full.n_components_} components')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"First 10 components explain {cumulative[9]:.2%} of variance")

In [ ]:
# 2D PCA projection for visualization
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# Scatter plot colored by digit class
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.6, s=30)
plt.colorbar(scatter, label='Digit Class')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA 2D Projection of Digits Dataset')
plt.show()
print("Note: Even with just 2 components, digit classes show partial separation")

## 🔍 2. Interpreting PCA – Loadings & Reconstruction

Understanding what each principal component represents helps interpret results. Loadings show how much each original feature (pixel) contributes to a component.

**Reconstruction**: $X_{recon} = X_{reduced} \cdot W_k^T + \mu$, where $W_k$ are the top-k eigenvectors. Fewer components = more compression, more loss.

In [ ]:
# Visualize loadings (components) as heatmaps
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    # Reshape 64-dimensional component back to 8x8 image
    loading = pca_all.components_[i].reshape(8, 8)
    im = ax.imshow(loading, cmap='RdBu_r', aspect='auto')
    ax.set_title(f'PC{i+1}')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Top 5 Principal Components as 8×8 Pixel Loadings', y=1.02)
plt.tight_layout()
plt.show()
print("Red = positive contribution, Blue = negative contribution to that component")

In [ ]:
# Reconstruct images using different numbers of components
def reconstruct_and_plot(n_components_list):
    fig, axes = plt.subplots(len(n_components_list)+1, 10, figsize=(15, 8))
    
    # Show original images in first row
    for digit in range(10):
        idx = np.where(y == digit)[0][0]  # First occurrence of each digit
        axes[0, digit].imshow(X[idx].reshape(8, 8), cmap='gray')
        axes[0, digit].set_title(f'Original {digit}')
        axes[0, digit].axis('off')
    
    # Reconstruct with different component counts
    for row, n_comp in enumerate(n_components_list, 1):
        pca_temp = PCA(n_components=n_comp)
        X_reduced = pca_temp.fit_transform(X_scaled)
        X_recon = pca_temp.inverse_transform(X_reduced)
        X_recon = scaler.inverse_transform(X_recon)  # Unscale to original range
        
        mse = np.mean((X - X_recon)**2)
        
        for digit in range(10):
            idx = np.where(y == digit)[0][0]
            axes[row, digit].imshow(X_recon[idx].reshape(8, 8), cmap='gray')
            if digit == 0:
                axes[row, digit].set_ylabel(f'{n_comp} comp\nMSE:{mse:.1f}', rotation=0, ha='right', va='center')
            axes[row, digit].axis('off')
    
    plt.suptitle('Reconstruction Quality vs Number of Components', y=0.98)
    plt.tight_layout()
    plt.show()

reconstruct_and_plot([5, 10, 20, 50])

## ⚡ 3. PCA as Preprocessing – Impact on Downstream Models

PCA is often used as a preprocessing step to speed up training and reduce overfitting. But does it always help? Let's compare raw vs PCA-reduced data on classification tasks.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_wine
import time

# Load wine dataset (13 features, 3 classes) for cleaner comparison
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
print(f"Wine dataset: {X_wine.shape[1]} features, {len(np.unique(y_wine))} classes")

# Prepare different feature sets
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# PCA variants
pca_10 = PCA(n_components=10)
X_wine_pca10 = pca_10.fit_transform(X_wine_scaled)

pca_95 = PCA(n_components=0.95)
X_wine_pca95 = pca_95.fit_transform(X_wine_scaled)
print(f"PCA(0.95) reduced wine to {pca_95.n_components_} components")

# Compare models
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

datasets = {
    'Raw (scaled)': X_wine_scaled,
    'PCA-10': X_wine_pca10,
    f'PCA-95% ({pca_95.n_components_} comp)': X_wine_pca95
}

results = []
for model_name, model in models.items():
    for data_name, X_data in datasets.items():
        start = time.time()
        scores = cross_val_score(model, X_data, y_wine, cv=5, scoring='accuracy')
        elapsed = time.time() - start
        results.append({
            'Model': model_name,
            'Data': data_name,
            'Accuracy': f"{scores.mean():.3f} ± {scores.std():.3f}",
            'CV Time (s)': f"{elapsed:.3f}",
            'Dimensions': X_data.shape[1]
        })

import pandas as pd
results_df = pd.DataFrame(results)
print("\nCross-validation results (Wine dataset):")
print(results_df.to_string(index=False))

## 🌐 4. t-SNE – Non-Linear Local Structure Visualization

t-SNE (t-Distributed Stochastic Neighbor Embedding) focuses on preserving local neighborhood structure. It converts high-D similarities to probabilities using Gaussian kernels, then minimizes KL divergence between high-D and low-D distributions.

**Key hyperparameter**: `perplexity` — roughly "effective number of neighbors" (typical 5-50). Too low = local clusters only; too high = loses local structure.

In [ ]:
from sklearn.manifold import TSNE

# t-SNE on digits (subset for speed)
np.random.seed(42)
subset_idx = np.random.choice(len(X), size=1000, replace=False)
X_subset = X_scaled[subset_idx]
y_subset = y[subset_idx]

# Default perplexity=30
tsne_default = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
X_tsne_default = tsne_default.fit_transform(X_subset)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne_default[:, 0], X_tsne_default[:, 1], c=y_subset, cmap='tab10', alpha=0.7, s=40)
plt.colorbar(scatter, label='Digit Class')
plt.title('t-SNE on Digits (perplexity=30)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.show()
print("t-SNE separates digit clusters clearly — notice the clean class boundaries")

In [ ]:
# Compare different perplexity values
perplexities = [5, 30, 50]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, n_jobs=-1)
    X_emb = tsne.fit_transform(X_subset)
    
    scatter = ax.scatter(X_emb[:, 0], X_emb[:, 1], c=y_subset, cmap='tab10', alpha=0.7, s=30)
    ax.set_title(f'Perplexity = {perp}')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE: Effect of Perplexity on Cluster Structure', y=1.02)
plt.tight_layout()
plt.show()
print("Low perplexity (5): tight local clusters, global structure lost")
print("High perplexity (50): more global structure, but clusters merge")
print("Sweet spot (30): balance of local and global preservation")

## 🚀 5. UMAP – Faster, Scalable, Better Global Structure

UMAP (Uniform Manifold Approximation and Projection) uses fuzzy simplicial sets and graph layout optimization. It's faster than t-SNE and often preserves more global structure.

**Key hyperparameters**:
- `n_neighbors`: Local vs global tradeoff (small = local structure, large = global structure)
- `min_dist`: Minimum distance between points in embedding (controls cluster tightness)

**Speed comparison**: UMAP is typically 2-5x faster than t-SNE on large datasets.

In [ ]:
# UMAP installation (uncomment if needed)
# !pip install umap-learn

import umap
import time

# UMAP with default parameters
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
start_umap = time.time()
X_umap = umap_model.fit_transform(X_subset)
time_umap = time.time() - start_umap

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_subset, cmap='tab10', alpha=0.7, s=40)
plt.colorbar(scatter, label='Digit Class')
plt.title(f'UMAP on Digits (n_neighbors=15, min_dist=0.1) — {time_umap:.2f}s')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.show()
print("UMAP preserves both local clusters AND relative global positioning of digit groups")

In [ ]:
# Speed comparison: t-SNE vs UMAP on larger subset
large_subset_idx = np.random.choice(len(X), size=3000, replace=False)
X_large = X_scaled[large_subset_idx]
y_large = y[large_subset_idx]

print(f"Timing on {len(X_large)} samples...")

# Time t-SNE
start = time.time()
tsne_large = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1).fit_transform(X_large)
time_tsne = time.time() - start

# Time UMAP
start = time.time()
umap_large = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_large)
time_umap_large = time.time() - start

print(f"t-SNE: {time_tsne:.2f}s | UMAP: {time_umap_large:.2f}s | Speedup: {time_tsne/time_umap_large:.1f}x")

# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
ax1.scatter(tsne_large[:, 0], tsne_large[:, 1], c=y_large, cmap='tab10', alpha=0.6, s=20)
ax1.set_title(f't-SNE ({time_tsne:.1f}s)')
ax1.set_xticks([])
ax1.set_yticks([])
ax2.scatter(umap_large[:, 0], umap_large[:, 1], c=y_large, cmap='tab10', alpha=0.6, s=20)
ax2.set_title(f'UMAP ({time_umap_large:.1f}s)')
ax2.set_xticks([])
ax2.set_yticks([])
plt.suptitle('Speed & Quality Comparison on 3000 samples')
plt.tight_layout()
plt.show()

## 📊 6. Comparing All Three on the Same Data

Let's visualize the fundamental differences: PCA (linear/global), t-SNE (non-linear/local), UMAP (non-linear/balanced). We'll use synthetic data with known structure.

In [ ]:
from sklearn.datasets import make_moons, make_blobs, make_circles

# Create challenging synthetic dataset: moons + blobs + noise
np.random.seed(42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.5)
X_circles, y_circles = make_circles(n_samples=300, factor=0.5, noise=0.05)

# Combine into 6D dataset (2D moons + 2D blobs + 2D circles)
X_combined = np.hstack([X_moons, X_blobs, X_circles])
y_combined = y_moons  # Use moons labels for coloring

print(f"Combined dataset shape: {X_combined.shape} — 6D data with non-linear structure")

# Standardize
X_comb_scaled = StandardScaler().fit_transform(X_combined)

# Apply all three methods
pca_comb = PCA(n_components=2).fit_transform(X_comb_scaled)
tsne_comb = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_comb_scaled)
umap_comb = umap.UMAP(n_components=2, n_neighbors=15, random_state=42).fit_transform(X_comb_scaled)

# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
methods = [('PCA', pca_comb), ('t-SNE', tsne_comb), ('UMAP', umap_comb)]

for ax, (name, embedding) in zip(axes, methods):
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=y_combined, cmap='viridis', alpha=0.7, s=50)
    ax.set_title(f'{name}')
    ax.set_xlabel(f'{name} 1')
    ax.set_ylabel(f'{name} 2')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Dimensionality Reduction Comparison: Linear vs Non-Linear Methods', y=1.02)
plt.tight_layout()
plt.show()
print("PCA: Linear projection may cross classes | t-SNE: Preserves local moon shape | UMAP: Balanced preservation")

## 🎯 7. Dimensionality Reduction for Clustering & Classification

Does preprocessing with PCA actually help downstream tasks? Let's test on clustering (KMeans) and classification (SVM/KNN).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.model_selection import train_test_split

# Use digits dataset for this experiment
X_dig = X_scaled
y_dig = y

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_dig, y_dig, test_size=0.3, random_state=42)

# Create PCA-reduced versions
pca_20 = PCA(n_components=20)
X_train_pca = pca_20.fit_transform(X_train)
X_test_pca = pca_20.transform(X_test)

print(f"Original dims: {X_train.shape[1]} | PCA dims: {X_train_pca.shape[1]}")

# Clustering comparison
kmeans_raw = KMeans(n_clusters=10, random_state=42, n_init=10).fit(X_train)
kmeans_pca = KMeans(n_clusters=10, random_state=42, n_init=10).fit(X_train_pca)

# Silhouette scores (higher = better defined clusters)
sil_raw = silhouette_score(X_train, kmeans_raw.labels_)
sil_pca = silhouette_score(X_train_pca, kmeans_pca.labels_)

print(f"\nClustering Silhouette Scores:")
print(f"  Raw data:     {sil_raw:.3f}")
print(f"  PCA-reduced:  {sil_pca:.3f}")
print(f"  Improvement:  {(sil_pca-sil_raw)/sil_raw:+.1%}")

In [ ]:
# Classification comparison
classifiers = {
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

clf_results = []
for name, clf in classifiers.items():
    # Train on raw data
    start = time.time()
    clf.fit(X_train, y_train)
    acc_raw = accuracy_score(y_test, clf.predict(X_test))
    time_raw = time.time() - start
    
    # Train on PCA data
    start = time.time()
    clf_pca = clf.__class__(**clf.get_params())  # Fresh instance
    clf_pca.fit(X_train_pca, y_train)
    acc_pca = accuracy_score(y_test, clf_pca.predict(X_test_pca))
    time_pca = time.time() - start
    
    clf_results.append({
        'Classifier': name,
        'Raw Acc': f"{acc_raw:.3f}",
        'PCA Acc': f"{acc_pca:.3f}",
        'Raw Time (s)': f"{time_raw:.3f}",
        'PCA Time (s)': f"{time_pca:.3f}",
        'Speedup': f"{time_raw/time_pca:.1f}x"
    })

clf_df = pd.DataFrame(clf_results)
print("\nClassification Results:")
print(clf_df.to_string(index=False))
print("\nKey insight: PCA often maintains accuracy while significantly speeding up training")

## ⚠️ Common Pitfalls & Pro Tips

Avoid these traps when applying dimensionality reduction:

- **Never use t-SNE/UMAP distances as features**: The distances in embedding space are not meaningful metrics — don't feed them into k-NN or clustering algorithms expecting Euclidean semantics
- **Perplexity sensitivity in t-SNE**: Values < 5 create artificial clusters; values > 50 merge distinct clusters. Always try 5, 30, 50 and compare
- **Crowding problem in t-SNE**: t-SNE cannot faithfully represent large intrinsic dimensionalities; it may collapse distinct clusters that are far apart in high-D space
- **Always scale before PCA/t-SNE/UMAP**: These methods are distance-based; unscaled features with different units will dominate variance calculations
- **Don't interpret non-convex clusters in PCA**: PCA is linear — if your data has a curved manifold, PCA will slice through it awkwardly
- **Random state matters for t-SNE/UMAP**: Different runs produce different layouts. Set `random_state` for reproducibility, or run multiple times to assess stability
- **Over-trusting 2D views**: A perfect 2D embedding doesn't mean your data is inherently 2D — check reconstruction error or trustworthiness metrics
- **Not checking explained variance in PCA**: Never blindly pick k components — verify cumulative variance or use `n_components=0.95`
- **UMAP n_neighbors too low**: Values < 5 focus only on very local structure, potentially fragmenting global clusters
- **Forgetting inverse_transform**: PCA can reconstruct data — use this to validate information loss visually
- **Applying t-SNE to huge datasets**: t-SNE is O(N²) — use UMAP or sample your data for large N (>10k)
- **Ignoring downstream task impact**: Always validate that reduction actually helps your final model, not just visualization

## 📝 Exercises

Test your understanding with these hands-on challenges:

**Exercise 1 (Easy)**: Apply PCA to the Iris dataset. Plot the explained variance ratio for all components and create a 2D scatter plot colored by species. How much variance do the first 2 components capture?

**Exercise 2 (Medium)**: On the digits dataset, find the minimum number of PCA components needed to capture 95% of variance. Reconstruct 5 sample images using only these components and compare to originals. Calculate the mean squared reconstruction error.

**Exercise 3 (Medium)**: Generate `make_circles` data with noise. Apply t-SNE with perplexity values of 5, 30, and 100, plus UMAP with n_neighbors values of 5, 15, and 50. Create a grid plot comparing all embeddings. Which method best preserves the circular topology?

**Exercise 4 (Hard)**: Load `fetch_california_housing`. Use PCA to reduce to 5, 10, and 20 components. Train a RandomForestRegressor on raw data vs each PCA variant. Compare RMSE and training time. Determine the optimal trade-off point where dimensionality reduction stops helping.

**Bonus (Exploratory)**: Load the wine dataset. Create a UMAP embedding colored by true class labels. Then run KMeans with k=3 on the UMAP coordinates and color by cluster labels. Calculate the adjusted Rand index between true labels and cluster assignments. Compare this to KMeans on raw scaled data.

<details>
<summary><b>Exercise Solutions (click to expand)</b></summary>

<br>

**Exercise 1 Solution:**
```python
from sklearn.datasets import load_iris
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

# Explained variance
pca_iris = PCA().fit(X_iris)
print(f"First 2 components: {pca_iris.explained_variance_ratio_[:2].sum():.1%}")

# 2D plot
X_iris_2d = PCA(n_components=2).fit_transform(X_iris)
plt.scatter(X_iris_2d[:, 0], X_iris_2d[:, 1], c=iris.target, cmap='viridis')
```

**Exercise 2 Solution:**
```python
pca_95 = PCA(n_components=0.95).fit(X_scaled)
n_comp = pca_95.n_components_
X_reduced = pca_95.transform(X_scaled)
X_recon = pca_95.inverse_transform(X_reduced)
X_recon_unscaled = scaler.inverse_transform(X_recon)
mse = np.mean((X - X_recon_unscaled)**2)
print(f"Components: {n_comp}, MSE: {mse:.2f}")
```

**Exercise 3 Solution:**
```python
X_circ, y_circ = make_circles(n_samples=500, noise=0.05, factor=0.5)
X_circ_scaled = StandardScaler().fit_transform(X_circ)
# Grid of t-SNE and UMAP plots varying perplexity/n_neighbors
# Best: UMAP with n_neighbors~15 preserves both circles
```

**Exercise 4 Solution:**
```python
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
# Compare raw vs PCA(5,10,20) on RMSE and training time
# Optimal point usually where PCA time savings outweigh accuracy loss
```

**Bonus Solution:**
```python
from sklearn.metrics import adjusted_rand_score
wine = load_wine()
X_w = StandardScaler().fit_transform(wine.data)
X_w_umap = umap.UMAP(random_state=42).fit_transform(X_w)
labels_true = wine.target
labels_kmeans_raw = KMeans(n_clusters=3, random_state=42).fit_predict(X_w)
labels_kmeans_umap = KMeans(n_clusters=3, random_state=42).fit_predict(X_w_umap)
print(f"ARI raw: {adjusted_rand_score(labels_true, labels_kmeans_raw):.3f}")
print(f"ARI UMAP: {adjusted_rand_score(labels_true, labels_kmeans_umap):.3f}")
```
</details>

## 🎓 Summary – What You Learned Today

- **PCA fundamentals**: Linear projection maximizing variance, eigenvectors as principal components, cumulative explained variance as quality metric
- **Interpretation skills**: Reading loadings to understand component composition, reconstructing data to assess information loss
- **Preprocessing applications**: PCA for speed vs accuracy trade-offs, effective for high-dimensional data with multicollinearity
- **t-SNE mastery**: Non-linear local structure preservation, perplexity tuning, understanding the crowding problem
- **UMAP advantages**: Scalability, speed, and better global structure preservation compared to t-SNE
- **Method selection**: PCA for preprocessing/compression, t-SNE for careful visualization of small datasets, UMAP for exploration of large datasets
- **Critical evaluation**: Always validate reduction impact on downstream tasks, never trust embedding distances blindly

---

## 🔮 Next Notebook Preview

**Notebook 16: Handling Imbalanced Data & Anomaly Detection**

When your classes are severely imbalanced (fraud detection, rare disease diagnosis) or you need to find the "weird" data points, standard ML breaks down. Next time we'll cover:

- **Resampling strategies**: Random oversampling/undersampling, SMOTE and its variants, ADASYN for synthetic minority generation
- **Algorithm-level fixes**: Class weights, cost-sensitive learning, threshold moving for optimal precision-recall balance  
- **Anomaly detection**: Isolation Forest for efficient outlier scoring, One-Class SVM for novelty detection, Local Outlier Factor for density-based anomalies
- **Evaluation metrics**: Why accuracy fails for imbalanced data, and how to properly use precision-recall curves, F1-score, and AUC-PR

*Get ready to handle the messy reality where positive cases are rare and outliers are the signal, not the noise.*